In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv("data/otto-group-product/train.csv")

# drop id
df = df.drop("id", axis=1)

X = df.drop("target", axis=1).values
y = df["target"].values


In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)   # 0..8

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


In [5]:
class OttoDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)  # long for CE loss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [6]:
class OttoDatasetTransform(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        if self.transform:
            x = self.transform(x)

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


In [7]:
def noise_transform(x):
    noise = np.random.normal(0, 0.01, size=x.shape)
    return x + noise


In [8]:
train_ds = OttoDatasetTransform(X_train, y_train, transform=noise_transform)
val_ds = OttoDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)


In [9]:
class OttoNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [10]:
num_classes = len(np.unique(y))

model = OttoNN(input_dim=X_train.shape[1], num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [11]:
def train_model(model, train_loader, val_loader, epochs=100, patience=10):
    
    best_val_loss = float("inf")
    patience_counter = 0
    
    for epoch in range(epochs):
        
        # ---- TRAIN ----
        model.train()
        train_loss = 0
        
        for Xb, yb in train_loader:
            logits = model(Xb)
            loss = criterion(logits, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # ---- VALIDATION ----
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for Xb, yb in val_loader:
                logits = model(Xb)
                loss = criterion(logits, yb)
                val_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1)
                correct += (preds == yb).sum().item()
                total += len(yb)
        
        val_loss /= len(val_loader)
        val_acc = correct / total
        
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        
        # ---- EARLY STOPPING ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = model.state_dict()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print("Early stopping triggered")
            break
    
    model.load_state_dict(best_weights)
    return model


In [12]:
model = train_model(model, train_loader, val_loader)


Epoch 1: train_loss=0.7420 val_loss=0.5732 val_acc=0.7778
Epoch 2: train_loss=0.5919 val_loss=0.5473 val_acc=0.7836
Epoch 3: train_loss=0.5648 val_loss=0.5317 val_acc=0.7921
Epoch 4: train_loss=0.5489 val_loss=0.5192 val_acc=0.7965
Epoch 5: train_loss=0.5310 val_loss=0.5089 val_acc=0.8010
Epoch 6: train_loss=0.5221 val_loss=0.5059 val_acc=0.8028
Epoch 7: train_loss=0.5131 val_loss=0.5045 val_acc=0.7999
Epoch 8: train_loss=0.5055 val_loss=0.4980 val_acc=0.8091
Epoch 9: train_loss=0.4959 val_loss=0.5026 val_acc=0.8026
Epoch 10: train_loss=0.4906 val_loss=0.4973 val_acc=0.8016
Epoch 11: train_loss=0.4828 val_loss=0.4938 val_acc=0.8065
Epoch 12: train_loss=0.4770 val_loss=0.4937 val_acc=0.8087
Epoch 13: train_loss=0.4756 val_loss=0.4921 val_acc=0.8090
Epoch 14: train_loss=0.4679 val_loss=0.4895 val_acc=0.8087
Epoch 15: train_loss=0.4671 val_loss=0.4870 val_acc=0.8081
Epoch 16: train_loss=0.4599 val_loss=0.4909 val_acc=0.8079
Epoch 17: train_loss=0.4598 val_loss=0.4857 val_acc=0.8100
Epoch 

In [13]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for Xb, yb in val_loader:
        logits = model(Xb)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == yb).sum().item()
        total += len(yb)

print("Validation Accuracy:", correct/total)


Validation Accuracy: 0.8158532643826761
